# sqrt-eps-stabilize — faded example 1: Place eps inside the sqrt, not outside

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sqrt-eps-stabilize`. Running the beacon reports progress on the `Numerical: sqrt-eps stabilization` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numerical: sqrt-eps stabilization` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sqrt-eps-stabilize`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sqrt-eps-stabilize"
DD_SUBTOPIC = "Numerical: sqrt-eps stabilization"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The critical distinction between `sqrt(v + eps)` and `sqrt(v) + eps` is about gradient stability near v=0. With eps inside, the gradient `d/dv sqrt(v + eps) = 1 / (2*sqrt(v+eps))` is bounded by `1/(2*sqrt(eps))` at v=0. With eps outside, `d/dv sqrt(v) = 1/(2*sqrt(v))` diverges to infinity at v=0. Both give finite forward values, but only the inside placement is safe during backpropagation.

## Faded exercise 1

Implement `stable_scale(g, v, eps=1e-8)` that returns the Adam-style scaled gradient `g / sqrt(v + eps)`.

1. Compute the denominator with eps inside the sqrt.
2. Divide g by the denominator.

The blank step is computing the denominator `sqrt(v + eps)`.

**Fill in:** Compute the stabilized denominator by taking the square root of (v + eps), placing eps inside the sqrt.

In [ ]:
import torch as t

t.manual_seed(0)

def stable_scale(g, v, eps=1e-8):
    denom = None  # TODO: Compute the stabilized denominator by taking the square root of (v + eps), placing eps inside the sqrt.
    return g / denom

g = t.tensor([1.0, 1.0, 1.0, 1.0])
v = t.tensor([0.0, 1e-9, 0.01, 1.0])

scaled = stable_scale(g, v)
print('v      :', v.tolist())
print('scaled :', [f'{x:.4f}' for x in scaled.tolist()])
print('all finite:', t.isfinite(scaled).all().item())
print('at v=0, scaled =', scaled[0].item(), '(should be 1/sqrt(eps) = 1e4)')


def _test():
    import torch as t
    import math

    g = t.tensor([1.0, 1.0, 1.0, 1.0])
    v = t.tensor([0.0, 1e-9, 0.01, 1.0])
    eps = 1e-8
    expected = g / t.sqrt(v + eps)
    scaled = stable_scale(g, v, eps)
    assert t.allclose(scaled, expected, atol=1e-6), f'scaled={scaled} expected={expected}'
    assert t.isfinite(scaled).all(), 'all values should be finite'
    # At v=0, result should be 1/sqrt(eps)
    expected_at_zero = 1.0 / math.sqrt(eps)
    assert abs(scaled[0].item() - expected_at_zero) < 1.0, f'at v=0: {scaled[0].item()}'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

def stable_scale(g, v, eps=1e-8):
    denom = t.sqrt(v + eps)
    return g / denom

g = t.tensor([1.0, 1.0, 1.0, 1.0])
v = t.tensor([0.0, 1e-9, 0.01, 1.0])

scaled = stable_scale(g, v)
print('v      :', v.tolist())
print('scaled :', [f'{x:.4f}' for x in scaled.tolist()])
print('all finite:', t.isfinite(scaled).all().item())
print('at v=0, scaled =', scaled[0].item())
```
</details>